# Pipeline LSTM para métricas de rede
Dataset com coluna obrigatória `Date` e demais colunas variáveis de métricas.
Previsão multivariada para instantes futuros com janela configurável.


## 1. Importação de bibliotecas


In [ ]:
!pip install numpy pandas matplotlib seaborn torch scikit-learn optuna plotly

In [ ]:
import copy, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import optuna
from optuna.samplers import TPESampler

sns.set_theme(style="whitegrid")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


## 2. Pré-processamento dos dados


In [ ]:
df = pd.read_csv("./dataset_lstm.csv")
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe()

### Feature extraction

In [ ]:
# Começamos retirando algumas colunas
df.drop(columns=["component_carrier_id", "rnti", "is_serving_cell"], inplace=True)
df.head()

In [ ]:
# Filtrando apenas para uma source
df = df[df.source == 'MMWAVE-PHY']
df.drop(columns='rsrq_db', inplace=True)
df.head()

In [ ]:
df.describe()

In [ ]:
df.reset_index(inplace=True)
df.info()

In [ ]:
# Valores únicos de IMSI, notamos que há um NaN que deve ser removido
df['imsi'].unique()

In [ ]:
# Removendo NaN
df = df[df.imsi.isna() == False]
df.drop(columns='index', inplace=True)
df.head()

In [ ]:
# Remodelando coluna de tempo para o formato datetime, exigido pelo pipeline
start = '2026-01-01'
df.time_s = pd.to_datetime(start) + pd.to_timedelta(df.time_s, unit='s')
df.head()

In [ ]:
# Normalizando valores de RSRP e SINR para balancear entradas

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df['normal_rsrp'] = scaler.fit_transform(df[['rsrp_dbm']])
df['normal_sinr'] = scaler.fit_transform(df[['sinr_db']])
df.drop(columns=['rsrp_dbm', 'sinr_db'], inplace=True)
df.head()

In [ ]:
# Criação das colunas separadas de RSRP e SINR: valores para cada gnB medida por tempo
for i in range(2, 5):
    df[f'RSRP_{i}'] = df[df['cell_id'] == float(i)]['normal_rsrp']
    df[f'SINR_{i}'] = df[df['cell_id'] == float(i)]['normal_sinr']

df.sort_values(by='time_s', inplace=True)
df.head()

In [ ]:
df.describe()

In [ ]:
# Para exemplo, vamos treinar apenas com os dados do UE 1 (imsi 1.0)

df1 = df[df.imsi == 1.]
df1.drop(columns=['source', 'imsi', 'normal_rsrp', 'normal_sinr'], inplace=True)

df1.head()

In [ ]:
# Excluímos a coluna 'cell_id', pois vamos unificar os valores por tempo
df1.drop('cell_id', inplace=True, axis=1)
df1.reset_index(drop=True, inplace=True)
df1.head()

In [ ]:
# Unificando os valores por tempo
df1 = df1.groupby('time_s', as_index=False).first()
df1

In [ ]:
# Verificando amostras que possuíam apenas RSRP/SINR de uma gnB e mantendo os que tem das 3 gnB
df1[(df1.RSRP_2.notna()) & (df1.RSRP_3.notna()) & (df1.RSRP_4.notna())].count()

In [ ]:
# Vamos manter apenas as amostras que possuem todas as colunas preenchidas.

df1 = df1[(df1.RSRP_2.notna()) & (df1.RSRP_3.notna()) & (df1.RSRP_4.notna())]
df1.reset_index(inplace=True, drop=True)
df1.head()

## Visualização

In [ ]:
# SINR e RSRP do gnB de ID 2

fig, ax = plt.subplots()
ax.plot(df1.time_s, df1.RSRP_2, label="RSRP")
ax.plot(df1.time_s, df1.SINR_2, label="SINR")

ax.set(xlabel='Tempo', ylabel='Valores (norm.)', title='Valor de RSRP e SINR da gnB 2')
ax.grid()

plt.legend()
plt.show()

# SINR e RSRP do gnB de ID 3

fig2, ax2 = plt.subplots()
ax2.plot(df1.time_s, df1.RSRP_3, label="RSRP")
ax2.plot(df1.time_s, df1.SINR_3, label="SINR")

ax2.set(xlabel='Tempo', ylabel='Valores (norm.)', title='Valor de RSRP e SINR da gnB 3')
ax2.grid()

plt.legend()
plt.show()

# SINR e RSRP do gnB de ID 4

fig3, ax3 = plt.subplots()
ax3.plot(df1.time_s, df1.RSRP_4, label="RSRP")
ax3.plot(df1.time_s, df1.SINR_4, label="SINR")

ax3.set(xlabel='Tempo', ylabel='Valores (norm.)', title='Valor de RSRP e SINR da gnB 4')
ax3.grid()

plt.legend()
plt.show()

In [ ]:
import plotly.io as pio

pio.renderers.default = "notebook"

In [ ]:
import plotly.express as px

# SINR e RSRP do gNB de ID 2
fig1 = px.line(
    df1,
    x="time_s",
    y=["RSRP_2", "SINR_2"],
    title="Valor de RSRP e SINR da gNB 2",
    labels={
        "time_s": "Tempo",
        "value": "Valores (norm.)",
        "variable": "Métrica",
    },
)
fig1.show()

# SINR e RSRP do gNB de ID 3
fig2 = px.line(
    df1,
    x="time_s",
    y=["RSRP_3", "SINR_3"],
    title="Valor de RSRP e SINR da gNB 3",
    labels={
        "time_s": "Tempo",
        "value": "Valores (norm.)",
        "variable": "Métrica",
    },
)
fig2.show()

# SINR e RSRP do gNB de ID 4
fig3 = px.line(
    df1,
    x="time_s",
    y=["RSRP_4", "SINR_4"],
    title="Valor de RSRP e SINR da gNB 4",
    labels={
        "time_s": "Tempo",
        "value": "Valores (norm.)",
        "variable": "Métrica",
    },
)
fig3.show()

Temos o caso em que o RSRP e o SINR aumentam em valor absoluto, mas a distribuição deles não muda. Isso nos leva a crer que o ambiente simulado está criando cenários pobres.

In [ ]:
# Enviamos esse novo dataset a um CSV
df1.to_csv('rsrp_sinr_triangulado-1.csv', index=False)

In [ ]:
DATA_PATH = "./rsrp_sinr_triangulado-1.csv"
DATE_COL = "time_s"
WINDOW_SIZE = 30
FORECAST_HORIZON = 3
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
RANDOM_SEED = 42
TARGET_ACCURACY_TOLERANCE = 0.10
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1) < 1e-8

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def load_data(path, date_col=DATE_COL):
    df = pd.read_csv(path)
    # if date_col not in df.columns: raise ValueError("Coluna obrigatória 'Date' não encontrada.")
    # df = df.copy(); df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    # df = df.dropna(subset=[date_col]).sort_values(date_col).reset_index(drop=True)
    metrics = [c for c in df.columns if c != date_col]
    if not metrics: raise ValueError("Inclua ao menos 1 métrica além de Date.")
    # df[metrics] = df[metrics].apply(pd.to_numeric, errors="coerce").interpolate(limit_direction="both").ffill().bfill()
    if df[metrics].isna().any().any(): raise ValueError("Ainda existem NaNs nas métricas.")
    return df, metrics

def make_seq(values, w, h):
    X, y, t = [], [], []
    for i in range(len(values) - w - h + 1):
        e = i + w; tg = e + h - 1
        X.append(values[i:e]); y.append(values[tg]); t.append(tg)
    return np.asarray(X, np.float32), np.asarray(y, np.float32), np.asarray(t, np.int64)

set_seed(RANDOM_SEED)
df, metric_cols = load_data(DATA_PATH, DATE_COL)
n = len(df)
if n <= WINDOW_SIZE + FORECAST_HORIZON: raise ValueError("Poucos dados para janela+horizonte.")
tr_end, va_end = int(n * TRAIN_RATIO), int(n * (TRAIN_RATIO + VAL_RATIO))

# scaler = StandardScaler(); scaler.fit(df.loc[:tr_end - 1, metric_cols].values)
# scaled = scaler.transform(df[metric_cols].values)
X, y, tidx = make_seq(scaled, WINDOW_SIZE, FORECAST_HORIZON)
dates = df[DATE_COL].iloc[tidx].reset_index(drop=True)

m_tr = tidx < tr_end; m_va = (tidx >= tr_end) & (tidx < va_end); m_te = tidx >= va_end
X_train, y_train, dates_train = X[m_tr], y[m_tr], dates.iloc[m_tr].reset_index(drop=True)
X_val, y_val, dates_val = X[m_va], y[m_va], dates.iloc[m_va].reset_index(drop=True)
X_test, y_test, dates_test = X[m_te], y[m_te], dates.iloc[m_te].reset_index(drop=True)
for nm, part in {"train":X_train, "val":X_val, "test":X_test}.items():
    if len(part) == 0: raise ValueError(f"Split {nm} vazio; ajuste janela/horizonte/proporções.")

print("Métricas:", metric_cols)
print("Shapes -> train", X_train.shape, y_train.shape, "| val", X_val.shape, y_val.shape, "| test", X_test.shape, y_test.shape)


In [ ]:
import plotly.express as px

# SINR e RSRP do gNB de ID 2
fig1 = px.line(
    df,
    x="time_s",
    y=["RSRP_2", "SINR_2"],
    title="Valor de RSRP e SINR da gNB 2",
    labels={
        "time_s": "Tempo",
        "value": "Valores (norm.)",
        "variable": "Métrica",
    },
)
fig1.show()

# SINR e RSRP do gNB de ID 3
fig2 = px.line(
    df,
    x="time_s",
    y=["RSRP_3", "SINR_3"],
    title="Valor de RSRP e SINR da gNB 3",
    labels={
        "time_s": "Tempo",
        "value": "Valores (norm.)",
        "variable": "Métrica",
    },
)
fig2.show()

# SINR e RSRP do gNB de ID 4
fig3 = px.line(
    df,
    x="time_s",
    y=["RSRP_4", "SINR_4"],
    title="Valor de RSRP e SINR da gNB 4",
    labels={
        "time_s": "Tempo",
        "value": "Valores (norm.)",
        "variable": "Métrica",
    },
)
fig3.show()

In [ ]:
df.describe()

In [ ]:
df1.describe()

In [ ]:
type(df.time_s.iloc[0])

## 3. Definição da LSTM (layers, etc.)


In [ ]:
def loader(X, y, bs, sh):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=bs, shuffle=sh)

class LSTMRegressor(nn.Module):
    def __init__(self, inp, hid, layers, drop, out):
        super().__init__()
        self.lstm = nn.LSTM(inp, hid, layers, batch_first=True, dropout=drop if layers > 1 else 0.0)
        self.fc = nn.Linear(hid, out)
    def forward(self, x):
        o, _ = self.lstm(x)
        return self.fc(o[:, -1, :])

def model_from(p):
    return LSTMRegressor(X_train.shape[2], p["hidden_size"], p["num_layers"], p["dropout"], y_train.shape[1]).to(DEVICE)

def run_epoch(model, dl, crit, opt=None):
    model.train() if opt else model.eval()
    loss_sum, preds, trues = 0.0, [], []
    for xb, yb in dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        if opt: opt.zero_grad()
        with torch.set_grad_enabled(opt is not None):
            out = model(xb); loss = crit(out, yb)
            if opt: loss.backward(); opt.step()
        loss_sum += loss.item() * xb.size(0)
        preds.append(out.detach().cpu().numpy()); trues.append(yb.detach().cpu().numpy())
    return loss_sum / len(dl.dataset), np.vstack(preds), np.vstack(trues)

def rmse_orig(pred_s, true_s):
    p = scaler.inverse_transform(pred_s); t = scaler.inverse_transform(true_s)
    return float(np.sqrt(mean_squared_error(t.reshape(-1), p.reshape(-1))))

def fit(model, dl_tr, dl_va, p, epochs, patience=15, warmup_epochs=20, delta=1e-4, verbose=False, trial=None, use_early_stopping=True):
    crit = nn.MSELoss(); opt = torch.optim.AdamW(model.parameters(), lr=p["lr"], weight_decay=p["weight_decay"])
    best, wait = 1e18, 0
    best_state = copy.deepcopy(model.state_dict())
    hist = {"train_loss": [], "val_loss": [], "val_rmse": []}
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5)
    
    for ep in range(1, epochs + 1):
        tr_loss, _, _ = run_epoch(model, dl_tr, crit, opt)
        hist["train_loss"].append(tr_loss)
        
        if dl_va is not None:
            va_loss, pva, tva = run_epoch(model, dl_va, crit)
            va_rmse = rmse_orig(pva, tva)
            scheduler.step(va_rmse)
            hist["val_loss"].append(va_loss); hist["val_rmse"].append(va_rmse)
            if va_rmse < best - delta: best, wait, best_state = va_rmse, 0, copy.deepcopy(model.state_dict())
            else:
                if ep > warmup_epochs:
                    wait += 1
            if verbose and (ep == 1 or ep % 10 == 0): print(f"ep {ep:03d} tr={tr_loss:.6f} va={va_loss:.6f} rmse={va_rmse:.6f}")
            if trial is not None:
                trial.report(va_rmse, ep)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()
            if use_early_stopping and (wait >= patience): 
                if verbose: 
                    print(f"Early stopping acionado na época {ep}.")
                break
                
    if dl_va is not None: model.load_state_dict(best_state)
    return model, hist, best

def predict(model, dl):
    model.eval(); p, t = [], []
    with torch.no_grad():
        for xb, yb in dl:
            out = model(xb.to(DEVICE))
            p.append(out.cpu().numpy()); t.append(yb.numpy())
    return np.vstack(p), np.vstack(t)


## 4. Otimização de hiperparâmetros com Optuna (menor RMSE)


In [ ]:
N_TRIALS = 20
OPTUNA_EPOCHS = 60
OPTUNA_PATIENCE = 8

def objective(trial):
    p = {
        "hidden_size": trial.suggest_int("hidden_size", 32, 256, step=32),
        "num_layers": trial.suggest_int("num_layers", 1, 3),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),
        "lr": 0.001,
        "weight_decay": trial.suggest_float("weight_decay", 1e-8, 1e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
    }
    dl_tr = loader(X_train, y_train, p["batch_size"], True)
    dl_va = loader(X_val, y_val, p["batch_size"], False)
    model = model_from(p)
    model, hist, best_rmse = fit(model, dl_tr, dl_va, p, OPTUNA_EPOCHS, OPTUNA_PATIENCE, verbose=False, trial=trial)
    trial.set_user_attr("epochs", len(hist["train_loss"]))
    return best_rmse

study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=RANDOM_SEED), pruner=optuna.pruners.MedianPruner(n_warmup_steps=5))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
best_params = study.best_trial.params
print("Melhor RMSE (val):", study.best_value)
print("Melhores parâmetros:", best_params)


In [ ]:
trials_df = study.trials_dataframe(attrs=("number", "value", "state"))
trials_ok = trials_df[trials_df["state"] == "COMPLETE"]
plt.figure(figsize=(10, 4))
sns.lineplot(data=trials_ok, x="number", y="value", marker="o")
plt.title("Optuna - RMSE por trial")
plt.xlabel("Trial")
plt.ylabel("RMSE (val)")
plt.show()


## 5. Treinamento da LSTM com os melhores hiperparâmetros (opcional)


In [ ]:
std_params = {'hidden_size': 64,
 'num_layers': 1,
 'dropout': 0.2,
 'lr': 0.0003,
 'weight_decay': 1e-04,
 'batch_size': 32}

In [ ]:
TRAIN_FINAL_MODEL = True
TRAIN_USING_OPT = False
USE_EARLY_STOPPING = True
REFIT_ON_TRAIN_PLUS_VAL = False
FINAL_EPOCHS = 120
FINAL_PATIENCE = 12

final_model, final_history = None, None

if TRAIN_FINAL_MODEL:
    if TRAIN_USING_OPT:
        p = best_params.copy()
        p["lr"] = 0.001
    else:
        p = std_params.copy()
    if REFIT_ON_TRAIN_PLUS_VAL:
        X_fit = np.concatenate([X_train, X_val], axis=0)
        y_fit = np.concatenate([y_train, y_val], axis=0)
        dl_fit = loader(X_fit, y_fit, p["batch_size"], True)
        final_model = model_from(p)
        crit = nn.MSELoss(); opt = torch.optim.AdamW(final_model.parameters(), lr=p["lr"], weight_decay=p["weight_decay"])
        final_history = {"train_loss": []}
        for ep in range(1, FINAL_EPOCHS + 1):
            tr_loss, _, _ = run_epoch(final_model, dl_fit, crit, opt)
            final_history["train_loss"].append(tr_loss)
            if ep == 1 or ep % 10 == 0: print(f"ep {ep:03d} train_loss={tr_loss:.6f}")
    else:
        dl_tr = loader(X_train, y_train, p["batch_size"], True)
        dl_va = loader(X_val, y_val, p["batch_size"], False)
        final_model = model_from(p)
        final_model, final_history, final_best = fit(final_model, dl_tr, dl_va, p, FINAL_EPOCHS, FINAL_PATIENCE, verbose=True, use_early_stopping=USE_EARLY_STOPPING, warmup_epochs=5)
        print("Melhor RMSE final (val):", final_best)
else:
    print("Treinamento final desativado.")


## 6. Avaliação com Acurácia, MAE, MSE, RMSE, MAPE, R² e gráficos


In [ ]:
def reg_acc(y_true, y_pred, tol=0.10):
    den = np.maximum(np.abs(y_true), 1e-8)
    return float(np.mean(np.abs((y_true - y_pred) / den) <= tol))

def mape(y_true, y_pred):
    den = np.where(np.abs(y_true) < 1e-8, 1e-8, np.abs(y_true))
    return float(np.mean(np.abs((y_true - y_pred) / den)) * 100)

def metrics_dict(y_true, y_pred, tol=0.10):
    yt, yp = y_true.reshape(-1), y_pred.reshape(-1)
    mse = mean_squared_error(yt, yp)
    return {
        "Acurácia": reg_acc(yt, yp, tol),
        "MAE": mean_absolute_error(yt, yp),
        "MSE": mse,
        "RMSE": float(np.sqrt(mse)),
        "MAPE (%)": mape(yt, yp),
        "R2": r2_score(yt, yp),
    }

if final_model is None: raise RuntimeError("Treine o modelo final primeiro.")

dl_test = loader(X_test, y_test, best_params["batch_size"], False)
pred_s, true_s = predict(final_model, dl_test)
pred = scaler.inverse_transform(pred_s)
true = scaler.inverse_transform(true_s)

m_global = metrics_dict(true, pred, TARGET_ACCURACY_TOLERANCE)
display(pd.DataFrame({"Métrica": list(m_global.keys()), "Valor": list(m_global.values())}))

rows = []
for i, c in enumerate(metric_cols):
    m = metrics_dict(true[:, i:i+1], pred[:, i:i+1], TARGET_ACCURACY_TOLERANCE)
    m["Métrica de rede"] = c
    rows.append(m)
per_metric = pd.DataFrame(rows)[["Métrica de rede", "Acurácia", "MAE", "MSE", "RMSE", "MAPE (%)", "R2"]]
display(per_metric)

plt.figure(figsize=(10,4))
plt.plot(final_history["train_loss"], label="Train Loss")
if "val_loss" in final_history and len(final_history["val_loss"]) > 0: plt.plot(final_history["val_loss"], label="Val Loss")
plt.title("Curva de Loss")
plt.xlabel("Época"); plt.ylabel("MSE"); plt.legend(); plt.show()

plot_n = len(metric_cols)
for i, c in enumerate(metric_cols[:plot_n]):
    plt.figure(figsize=(12,4))
    plt.plot(pd.to_datetime(dates_test), true[:, i], label="Real")
    plt.plot(pd.to_datetime(dates_test), pred[:, i], label="Previsto", alpha=0.85)
    plt.title(f"Real vs Previsto - {c}")
    plt.xlabel("Data"); plt.ylabel(c); plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(6,6))
sns.scatterplot(x=true.reshape(-1), y=pred.reshape(-1), alpha=0.35, s=20)
mn, mx = min(true.min(), pred.min()), max(true.max(), pred.max())
plt.plot([mn, mx], [mn, mx], "r--")
plt.title("Dispersão Global: Real vs Previsto")
plt.xlabel("Real"); plt.ylabel("Previsto"); plt.show()

plt.figure(figsize=(10,4))
sns.histplot((true - pred).reshape(-1), bins=40, kde=True)
plt.title("Distribuição dos Resíduos")
plt.xlabel("Resíduo"); plt.ylabel("Frequência"); plt.show()


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

nome = "relatorio_treinamento16.pdf"

def add_table_to_pdf(pdf, df, title=""):
    fig, ax = plt.subplots(figsize=(max(8, len(df.columns) * 1.5), len(df) * 0.4 + 1.5))
    ax.axis('off')
    if title:
        plt.title(title, fontsize=12, weight='bold', pad=10)
    table = ax.table(cellText=np.round(df.values, 4) if np.issubdtype(df.values.dtype, np.number) else df.values, 
                     colLabels=df.columns, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.2, 1.2)
    plt.tight_layout()
    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

with PdfPages(nome) as pdf:
    # 1. Tabelas
    df_global = pd.DataFrame({"Métrica": list(m_global.keys()), "Valor": list(m_global.values())})
    df_hiperparam = pd.DataFrame(p, index=[0])
    add_table_to_pdf(pdf, df_hiperparam, "Conjunto de Hiperparâmetros")
    add_table_to_pdf(pdf, df_global, "Métricas Globais")
    add_table_to_pdf(pdf, per_metric, "Métricas por Métrica de Rede")
    
    # 2. Curva de Loss
    fig = plt.figure(figsize=(10,4))
    plt.plot(final_history["train_loss"], label="Train Loss")
    if "val_loss" in final_history and len(final_history["val_loss"]) > 0: 
        plt.plot(final_history["val_loss"], label="Val Loss")
    plt.title("Curva de Loss")
    plt.xlabel("Época"); plt.ylabel("MSE"); plt.legend(); plt.tight_layout()
    pdf.savefig(fig); plt.close()

    # 3. Real vs Previsto por Coluna
    plot_n = min(4, len(metric_cols))
    for i, c in enumerate(metric_cols[:plot_n]):
        fig = plt.figure(figsize=(12,4))
        plt.plot(pd.to_datetime(dates_test), true[:, i], label="Real")
        plt.plot(pd.to_datetime(dates_test), pred[:, i], label="Previsto", alpha=0.85)
        plt.title(f"Real vs Previsto - {c}")
        plt.xlabel("Data"); plt.ylabel(c); plt.legend(); plt.tight_layout()
        pdf.savefig(fig); plt.close()

    # 4. Dispersão Global
    fig = plt.figure(figsize=(6,6))
    sns.scatterplot(x=true.reshape(-1), y=pred.reshape(-1), alpha=0.35, s=20)
    mn, mx = min(true.min(), pred.min()), max(true.max(), pred.max())
    plt.plot([mn, mx], [mn, mx], "r--")
    plt.title("Dispersão Global: Real vs Previsto")
    plt.xlabel("Real"); plt.ylabel("Previsto"); plt.tight_layout()
    pdf.savefig(fig); plt.close()

    # 5. Distribuição dos Resíduos
    fig = plt.figure(figsize=(10,4))
    sns.histplot((true - pred).reshape(-1), bins=40, kde=True)
    plt.title("Distribuição dos Resíduos")
    plt.xlabel("Resíduo"); plt.ylabel("Frequência"); plt.tight_layout()
    pdf.savefig(fig); plt.close()

print(f"Relatório salvo com sucesso como '{nome}'")